# Day 18 — Handling Missing Values in Pandas

This notebook covers Detection (`isna()`), Type Upcasting (`np.nan` vs `Int64`), Drop Strategies (`dropna()`), Statistical/Constant Imputation (`fillna()`), and Sequential Time-Series Fill (`ffill()`)

## Validation goals
- Distinguish between missing data representations (`np.nan`, `None`, and nullable `pd.NA`)[cite: 15]
- Detect and quantify missing value patterns using `df.isna().sum()`[cite: 15]
- Master conditional dropping strategies (`dropna(subset=...)`, `how='any'` vs `how='all'`)[cite: 15]
- Execute statistical, constant, and sequential forward-fill (`ffill()`) imputations[cite: 15]
- Analyze type upcasting penalties (`int64` → `float64`) and leverage modern nullable extension types (`Int64`)[cite: 15]

Real-world datasets are rarely pristine. Missing data—arising from sensor dropouts, API failures, or unrecorded user fields—can break downstream machine learning models and corrupt aggregations if handled incorrectly.

Pandas provides robust tooling to detect, quantify, drop, or impute missing values. Understanding the nuances of how missingness is represented and how operations handle `NaN` (Not a Number) is essential for building resilient data pipelines.

### Core Concepts & Deep Dive

1. **Representation of Missing Data**
    Pandas handles missing data using a few distinct representations depending on the data type:

    - `np.nan` (`Float NaN`): The standard representation for missing numerical data. Because `NaN` is technically a floating-point type, introducing a missing value into an integer column forces Pandas to upcast the entire column from `int64` to `float64`.

    - `None` (`Python Object`): Used in `object-dtype` columns (typically strings).

    - `pd.NA` (`Nullable Extension Dtypes`): Introduced in modern Pandas to support missing values in integer, boolean, and string columns without forcing float upcasting.

2. **Detection and Quantification**
    Before cleaning data, you must locate missing values:

    - `df.isna() / df.isnull()`: Returns a boolean DataFrame of the same shape indicating True where values are missing.

    - Quantification: Combining methods like `df.isna().sum()` provides the exact count of missing entries per column, while `.mean()` reveals the missingness proportion.

3. **Deletion Strategies `(dropna())`**
    Dropping rows or columns containing missing data should be done carefully to avoid discarding valuable information:

    how='any' vs how='all': how='any' drops a row/column if any value is missing. how='all' drops it only if every value is missing.

    subset parameter: Restricts the check to specific columns, ensuring rows are not dropped due to missing values in irrelevant fields.

4. **Imputation Strategies `(fillna())`**
    Instead of discarding rows, missing values can be filled with estimated or calculated values:

    - Constant Replacement: Filling with a static value (e.g., `0`, `"UNKNOWN"`).

    - Statistical Imputation: Filling numeric columns with column-wise statistics such as the mean `(df['col'].mean())` or `median`.

    - Sequential/Interpolation Imputation `(ffill, bfill)`: In time-series or ordered data, forward-fill (`method='ffill'`) propagates the last valid observation forward, while backward-fill (`method='bfill'`) uses the next valid observation.


**Practical Implementation: Missing Value Pipeline**
Here is a comprehensive script demonstrating detection, conditional dropping, and multi-strategy imputation on a messy dataset:

In [1]:
import numpy as np
import pandas as pd

# Setup dataset with intentionally injected missing values
np.random.seed(42)
n_rows = 10

raw_data = {
    "sensor_id": [f"SENS_{i:03d}" for i in range(n_rows)],
    "temperature": [22.5, np.nan, 23.1, 21.8, np.nan, 24.0, 22.1, np.nan, 23.5, 22.9],
    "pressure_kpa": [101.3, 101.2, np.nan, np.nan, 100.9, np.nan, 101.1, 101.0, np.nan, 101.4],
    "status_code": ["OK", "WARN", None, "OK", "ERR", "OK", None, "OK", "WARN", "OK"]
}

df = pd.DataFrame(raw_data)

print("--- Original DataFrame with Missing Values ---")
print(df)
print("\nMissing Value Counts per Column:")
print(df.isna().sum())

# ==========================================
# 1. Dropping Strategy (Selective Subset)
# ==========================================
# Drop rows only if 'temperature' is missing, preserving rows with other missing fields
df_dropped = df.dropna(subset=["temperature"])
print(f"\n--- After dropping rows where 'temperature' is NaN (Rows: {len(df_dropped)}) ---")
print(df_dropped)


# ==========================================
# 2. Imputation: Statistical & Constant Fill
# ==========================================
# Impute numerical 'temperature' with its mean
mean_temp = df["temperature"].mean()
df_imputed = df.copy()
df_imputed["temperature"] = df_imputed["temperature"].fillna(mean_temp)

# Impute categorical 'status_code' with a constant string
df_imputed["status_code"] = df_imputed["status_code"].fillna("UNKNOWN")

print("\n--- After Statistical & Constant Imputation ---")
print(df_imputed[["sensor_id", "temperature", "status_code"]])


# ==========================================
# 3. Imputation: Sequential Forward-Fill (ffill)
# ==========================================
# Useful for time-series / ordered sensor telemetry
df_forward_filled = df.copy()
df_forward_filled["pressure_kpa"] = df_forward_filled["pressure_kpa"].ffill()

print("\n--- After Forward-Fill on Pressure Column ---")
print(df_forward_filled[["sensor_id", "pressure_kpa"]])

--- Original DataFrame with Missing Values ---
  sensor_id  temperature  pressure_kpa status_code
0  SENS_000         22.5         101.3          OK
1  SENS_001          NaN         101.2        WARN
2  SENS_002         23.1           NaN         NaN
3  SENS_003         21.8           NaN          OK
4  SENS_004          NaN         100.9         ERR
5  SENS_005         24.0           NaN          OK
6  SENS_006         22.1         101.1         NaN
7  SENS_007          NaN         101.0          OK
8  SENS_008         23.5           NaN        WARN
9  SENS_009         22.9         101.4          OK

Missing Value Counts per Column:
sensor_id       0
temperature     3
pressure_kpa    4
status_code     2
dtype: int64

--- After dropping rows where 'temperature' is NaN (Rows: 7) ---
  sensor_id  temperature  pressure_kpa status_code
0  SENS_000         22.5         101.3          OK
2  SENS_002         23.1           NaN         NaN
3  SENS_003         21.8           NaN          OK
5  

### Guided Question:

1: **Type Upcasting and NaN**
Suppose you have a DataFrame column containing integer counts `([10, 20, 30])`. When you introduce a missing value `(np.nan)` into that column, Pandas automatically converts the column `dtype` from `int64` to `float64`.

Why does standard `NaN` force this type conversion, and how do modern nullable extension dtypes (e.g., `Int64` with `capital I`) prevent integer columns from turning into `floats`?

The reason is that standard np.nan is a floating-point value, and traditional NumPy integer types such as int64 cannot represent missing values.
1. Why does NaN force the conversion?
Consider this example:

In [3]:
import pandas as pd
import numpy as np

df = pd.DataFrame({"count": [10, 20, 30]})

print(df["count"].dtype)  # int64

df.loc[1, "count"] = np.nan

print(df["count"].dtype)  # float64

int64
float64


When you introduce np.nan, Pandas typically converts the integer column to `float64` because:
- Standard NumPy int64 has no dedicated representation for missing values.
- `np.nan` is represented as a `floating-point` value.
- `float64` can represent both ordinary `integer-valued` numbers and `NaN`.
The resulting column contains values equivalent to `10.0, NaN, and 30.0`.

2. How does the nullable Int64 dtype solve this?
`Pandas` provides a nullable integer extension `dtype` called `Int64` `(capital I)`. Unlike `NumPy's` standard `int64`, it supports missing values using a separate missing-value mask.

In [2]:
import pandas as pd

df = pd.DataFrame({
    "count": pd.Series([10, 20, 30], dtype="Int64")
})

df.loc[1, "count"] = pd.NA

print(df)
print(df["count"].dtype)

   count
0     10
1   <NA>
2     30
Int64


Important distinction: `int64` and `Int64` are different dtypes. The `capital I `identifies `Pandas`' nullable extension type.

Key takeaway: Use `Int64` when a column represents whole-number counts or `IDs` and may contain missing values. This preserves integer semantics instead of introducing `floating-point` values just to accommodate missing data.

### Guided Question:

**Time-Series Imputation Choices**
When analyzing chronological financial tick data or `IoT` sensor streams, missing values frequently occur over short time intervals.

Why is using global statistical imputation (like filling missing sensor readings with the overall column mean) dangerous in ordered time-series data, and why is forward-fill `(ffill())` or interpolation typically preferred?

Global statistical imputation, such as replacing missing time-series readings with the overall column mean, can be dangerous because it ignores the chronological order and temporal patterns of the data.
1. Why global mean imputation can be misleading
Suppose an IoT sensor records temperature throughout a day:
Time	Temperature
08:00	18°C
09:00	20°C
10:00	Missing
11:00	24°C
12:00	26°C
If you fill the missing value with the overall mean of 22°C, you introduce a value that doesn't accurately reflect the likely temperature at 10:00. The surrounding observations suggest it was closer to 22°C in this example, but that won't always be true.

The major risks are:
- Loss of temporal patterns: Global means ignore trends, seasonality, and local fluctuations.
- Artificial smoothing: Imputed values can reduce observed variance and distort statistical relationships.
- Look-ahead bias: If the overall mean uses future observations, it can leak future information into a model that should only use past data. This is particularly dangerous in financial forecasting.
- Misleading downstream analysis: Distorted values can affect anomaly detection, volatility estimates, and predictive models.

2. Why use forward-fill or interpolation?
Forward-fill `(ffill())`
Forward-fill uses the most recent observed value to fill a subsequent gap.

`df["temperature"] = df["temperature"].ffill()`

This is useful when values are expected to remain stable between observations, such as a device's last reported status or a slowly changing configuration.

Example: `18, 20, NaN, 24 → 18, 20, 20, 24`
Interpolation
Interpolation estimates missing values using surrounding observations. For numeric data with regular time intervals, linear interpolation is a common choice.

`df["temperature"] = df["temperature"].interpolate(method="linear")`

Example: `18, 20, NaN, 24 → 18, 20, 22, 24`

Interpolation is often appropriate when a variable changes relatively smoothly over time.

**Situation:**	**Potentially suitable method**


**Slowly changing sensor readings:**	Forward-fill

**Smoothly changing measurements**	Linear interpolation

**Financial prices with brief gaps:**	Depends on the analysis; forward-fill may suit price-state alignment

**Financial returns or irregular tick data:**	Avoid automatic filling without considering market events and timestamps

**Long periods of missing observations:**	Investigate the cause before imputing

One important caveat: forward-fill and interpolation are not universally better. Forward-fill can create artificial flat periods, while interpolation can invent values that were never observed. Linear interpolation also uses future observations when filling an interior gap, so it is generally unsuitable for a strictly real-time prediction pipeline unless that information would genuinely be available.

Key takeaway: Time-series imputation should preserve the temporal structure of the data as much as possible. Choose the method based on how the variable behaves over time, why the data is missing, and whether future observations are allowed to influence the estimate.

### Guided Question:

**Evaluating dropna() Parameters**
Consider a large DataFrame where you need to clean records. You apply `df.dropna(how='all')` versus `df.dropna(how='any')`.

Under what structural conditions would `how='all'` remove zero rows even if numerous individual cells contain missing values?

`df.dropna(how='all')` removes only rows where every cell is missing (NaN or pd.NA).

Therefore, it would remove zero rows if every row contains at least one non-missing value, even when numerous individual cells contain missing values.

For example:

In [4]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name": ["Ahmad", None, "Fatima"],
    "age": [np.nan, 25, np.nan],
    "city": ["Lagos", np.nan, "Abuja"]
})

cleaned = df.dropna(how="all")

print(cleaned)

     name   age   city
0   Ahmad   NaN  Lagos
1     NaN  25.0    NaN
2  Fatima   NaN  Abuja


No rows are removed because every row has at least one valid value.
- `how='all'`: Drops a row only if every column is missing.
- `how='any'`: Drops a row if at least one column is missing.
Key takeaway: `how='all'` removes zero rows when no row is entirely missing, regardless of how many individual cells contain missing values.